# README

- **Author**: 詹明翰
- **Created At**: 2026-10-16
- **Last Modified At**: 2026-10-16

---
## What does this file do?

準備三張敘述性統計表的輸入。保留全部原始列與欄位，不去重、不補值；增加紀錄流水號、
日期時間、行政區、PDF 中文代碼標籤、死亡指標，以及依事故座標計算的日出日落與夜間指標。
全部列皆為第一當事人；完整列與「年月日時分＋原始地點＋座標」候選鍵沒有重複。
每列暫作一件事故，仍缺少官方事故識別碼，不宣稱候選鍵已獲官方驗證。

夜間為當地時間早於當日日出，或晚於當日日落；等於邊界不列入。
使用 Astral 3.2、原始座標、海拔 0 公尺、Asia/Taipei。座標缺失則保留夜間缺值，
不代入市中心座標。另產生固定 18:49（含）至 05:02（不含）的敏感度指標。
死亡指標沿用使用者指定的「死亡人數 ≥ 1」，代表 24 小時內死亡；2–30 日死亡另列。
速限保留原值，包括 0；不自行將 0 解讀為缺值。

---
## What does this file take?

- **Source Data Sets**:
  1. `data/raw/114年-臺北市死傷交通事故明細.csv`；Input key: `crashes`；22,762 列、47 欄。
     原始資料索引見 [PROJECT.md](../../PROJECT.md#data-artifacts)。
     SHA-256: `61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c`。
  2. [事故對照表.pdf](../raw/事故對照表.pdf)（第 1 頁道路照明、天候、道路型態；112.07.01 後適用）。
     SHA-256: `f40ded1ded1c816a38f421bc972cdc1320bbd280e647792207ec037d05fd3670`。
- 執行環境：`python -m pip install -r data/code/descriptive_requirements.txt`（從專案根目錄執行）。

---
## What does this file output?

- `data/processed/taipei_2025_descriptive.csv`；Output key: `descriptive`；Role: `analysis`。
- Documentation: [資料說明](../processed/taipei_2025_descriptive_documentation.ipynb)。
- Expected schema and row-count rules: 原始 47 欄不變，增加 11 欄，保留 22,762 列；紀錄流水號唯一。
- Verification status / SHA-256: 執行末格重新讀檔核對，實際結果見輸出。
- Downstream finding input: `data/processed/taipei_2025_descriptive.csv`。

原始資料不變動。可重跑並更新本 Notebook 所擁有的同名衍生檔。

## 閱讀順序與資料流程

本 Notebook 負責把原始事故資料轉為分析可以直接讀取的資料。
下方依序做五件事：設定讀檔工具、核對來源、建立代碼對照、產生分析欄位、保存並驗證結果。
可以將它理解為「原始 CSV → 保留原欄位並增加分析欄位 → 衍生 CSV」。
統計表則放在 [正式分析 Notebook](../../finding/敘述性統計.ipynb)。

### 1. 載入套件與確認專案路徑

`pandas` 負責表格處理，`numpy` 協助產生流水號，`hashlib` 用來辨識檔案版本。
程式會從目前位置往上尋找專案根目錄，因此從 `data/code/` 開啟也能找到正確輸入。
SHA-256 的作用是確認讀到與本次核對相同的檔案；它本身不能證明資料內容正確。

In [1]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

def find_project_root():
    start = Path.cwd().resolve()
    for p in (start, *start.parents):
        if (p / 'AGENTS.md').is_file() and (p / 'templates').is_dir():
            return p
    raise FileNotFoundError('請從含 AGENTS.md 與 templates/ 的專案內開啟 Notebook。')

PROJECT_ROOT = find_project_root()

def project_file(relative, area, must_exist=True):
    rel = Path(relative)
    if rel.is_absolute() or '..' in rel.parts:
        raise ValueError('請使用精確的專案相對路徑。')
    path = (PROJECT_ROOT / rel).resolve()
    if not path.is_relative_to((PROJECT_ROOT / 'data' / area).resolve()):
        raise ValueError('資料路徑超出預期目錄。')
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def verify(path, expected):
    actual = sha256_file(path)
    if actual != expected:
        raise ValueError(f'檔案版本改變，請核對後再執行：{path.name}')
    return actual

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 15)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

### 2. 核對來源與每列代表的紀錄

先檢查 CSV 與 PDF 的 SHA-256，再確認列數、欄數與當事人序號。
`duplicated()` 同時檢查完整列及候選事故鍵；候選鍵使用日期時間、地點與座標，
用於找出可能重複的紀錄。本次沒有發現這兩種重複，因此不進行刪除。
同一地點在不同時間發生的事故可以各自存在，不能只憑地點相同就去重。

In [2]:
raw_path = project_file('data/raw/114年-臺北市死傷交通事故明細.csv', 'raw')
pdf_path = project_file('data/raw/事故對照表.pdf', 'raw')
verify(raw_path, '61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c')
verify(pdf_path, 'f40ded1ded1c816a38f421bc972cdc1320bbd280e647792207ec037d05fd3670')
raw = pd.read_csv(raw_path, encoding='utf-8-sig')
assert raw.shape == (22762, 47)
assert raw['當事人序號'].eq(1).all()
candidate_key = ['發生年度','發生月','發生日','發生時-Hours','發生分','肇事地點','座標-X','座標-Y']
assert not raw.duplicated().any()
assert not raw.duplicated(candidate_key).any()
print('原始列數：', len(raw), '；完整列及候選鍵重複：0；未刪除任何列。')

原始列數： 22762 ；完整列及候選鍵重複：0；未刪除任何列。


### 3. 將數字代碼轉為可閱讀的標籤

光線、天候與道路型態的數字代表類別。例如光線 5、6、7 分別表示設備開啟、
未開啟或故障、無設備；數字較大不代表照度比較高或風險比較大。
對照來自 PDF 第 1 頁，原始代碼欄仍保留，以便日後追溯。
程式也會檢查是否出現未收錄的代碼，避免將未知代碼悄悄轉成缺值。

In [3]:
LIGHT = {1:'日間自然光線', 2:'晨或暮光', 3:'夜間（隧道、地下道、涵洞）有照明',
         4:'夜間（隧道、地下道、涵洞）無照明', 5:'有照明且開啟',
         6:'有照明未開啟或故障', 7:'無照明'}
WEATHER = {1:'暴雨', 2:'強風', 3:'風沙', 4:'霧或煙', 5:'雪', 6:'雨', 7:'陰', 8:'晴', 9:'風'}
ROAD = {1:'有遮斷器平交道', 2:'無遮斷器平交道', 3:'三岔路', 4:'四岔路', 5:'多岔路',
        6:'隧道', 7:'地下道', 8:'橋樑', 9:'涵洞', 10:'高架道路', 11:'彎曲路及附近',
        12:'坡路', 13:'巷弄', 14:'直路', 18:'圓環', 19:'廣場',
        20:'休息站或服務區', 21:'輕軌共用道路', 22:'其他'}

### 4. 建立時間、中文類別與死亡／夜間指標

民國年加上 1911 後與月、日、時、分組成事故時間。`record_id` 僅記錄原始列順序，
不視為官方事故編號；行政區名稱則移除原始區序開頭的數字。

`fatal24` 將「死亡人數 ≥ 1」轉為 1，已知沒有死亡轉為 0；原欄缺值仍是未知。
因此它的平均數可用來表示有效紀錄中的死亡事故比例，而不是每件平均死亡人數。

夜間主要指標使用每筆事故的經緯度計算當日日出與日落，
再判斷事故是否早於日出或晚於日落。這能反映季節差異；座標缺失時無法做相同判斷。
`night_fixed` 另外使用固定鐘點，供後續比較結果是否依賴夜間定義。
下方前十筆僅供檢查欄位建立是否合理，不能當成全年度分布。

## 夜間判定的計算來源與定義

**資料來源與方法。** 本研究沒有將中央氣象署公布的日出日落表匯入資料。
日出與日落時間由 Python 的 **Astral 3.2** 計算：輸入每筆事故的西元日期、
`座標-Y`（緯度）與 `座標-X`（經度），海拔統一設為 0 公尺，使用臺灣時間 UTC+8
（2025 年的 Asia/Taipei）。套件依天文模型推算太陽位置及日出日落，並非現場亮度量測。

**主要指標 `is_night`。** 令事故的當地時間為 t、該日該位置的日出與日落為 R、S：
當 **t < R 或 t > S** 時設為 1，其餘為 0；恰好等於日出或日落歸為 0。
若經度或緯度缺失，日出日落及主要夜間指標都保留缺值，不用市中心座標代替。
本資料有 2 筆這類紀錄。`光線` 是照明設備狀態，不能直接取代這個日夜判斷。

**固定指標 `night_fixed`。** 依本次確認的研究設計，採「全年最晚日落之後、
全年最早日出之前」作為保守的固定時段。先對 **2025 年完整 365 日**，
在資料全部有效座標的經緯度外接矩形四角計算日出日落，涵蓋本資料中的位置差異；
經度範圍為 121.4611803–121.6520178、緯度為 24.9659305–25.1832948。
這是本資料的空間範圍，並非臺北市官方行政邊界。

四角全年計算的最早日出為 **2025-06-09 05:02:50.800739**（東北角），
最晚日落為 **2025-07-02 18:48:12.258035**（西北角）。配合事故時間只有分鐘精度，
將最早日出向下取整為 05:02、最晚日落向上取整為 18:49。
故 `night_fixed = 1` 的時段為 **18:49（含）至翌日 05:02（不含）**，
即一天中的分鐘數 m ≥ 1129 或 m < 302，其餘設為 0。
另外逐筆驗證：所有有有效座標且被固定時段納入的事故，必須同時符合其當日 `is_night = 1`。

這個固定指標是較窄的夜間子樣本；值為 0 只表示不在固定時段內，**不代表白天**。
固定時段依時間即可套用；但缺座標者無法逐筆驗證實際位置，所以「皆為夜間」的檢查
限定於有有效座標的紀錄，不能延伸到其他年度、其他地區或未知位置。

**為何不只採夏至？** 夏至白晝最長，並不保證當天是全年日出最早或日落最晚。
此點參考 [臺北市立天文科學教育館的 2025 夏至說明](https://tam.gov.taipei/News_Content.aspx?n=B64052C7930D4913&s=C881A8681CC67490&sms=2CF1F5E2E0B96411)。
該網頁用於確認天文概念；本資料各筆時間與固定界線仍由 Astral 計算，不冒稱為官方公布值。

**解讀限制。** 日落後與日出前可能仍有暮光或曙光；天候、山勢、建築遮蔽與實際照明
也會影響亮度。因此本定義表示模型下的日出日落夜間，不能稱為「完全黑暗」。

In [4]:
# 優先使用已安裝套件；亦支援本次工作區的暫存依賴目錄。
try:
    from astral import Observer
    from astral.sun import sunrise, sunset
except ImportError:
    sys.path.insert(0, str(PROJECT_ROOT / 'data/temp/python-deps'))
    from astral import Observer
    from astral.sun import sunrise, sunset
from importlib.metadata import version
assert version('astral') == '3.2', '請依 requirements 安裝 Astral 3.2。'
from datetime import timezone, timedelta
tz = timezone(timedelta(hours=8))  # 臺灣 2025 年全年 UTC+8


from itertools import product
import math

years = sorted((raw['發生年度'] + 1911).unique())
assert years == [2025], '固定時段適用年度改變，請重新核對研究設計。'
lon_bounds = (float(raw['座標-X'].min()), float(raw['座標-X'].max()))
lat_bounds = (float(raw['座標-Y'].min()), float(raw['座標-Y'].max()))
boundary_rows=[]
def minute_of_day(timestamp):
    return timestamp.hour*60 + timestamp.minute + timestamp.second/60 + timestamp.microsecond/60000000

for lat, lon in product(lat_bounds,lon_bounds):
    observer=Observer(latitude=lat,longitude=lon,elevation=0)
    for date in pd.date_range('2025-01-01','2025-12-31'):
        rise=sunrise(observer,date=date.date(),tzinfo=tz)
        setting=sunset(observer,date=date.date(),tzinfo=tz)
        boundary_rows.append({'日期':str(date.date()),'緯度':lat,'經度':lon,
                              '日出':rise.isoformat(),'日落':setting.isoformat(),
                              '日出分鐘':minute_of_day(rise),'日落分鐘':minute_of_day(setting)})
boundary=pd.DataFrame(boundary_rows)
assert len(boundary)==4*365
earliest=boundary.loc[boundary['日出分鐘'].idxmin()]
latest=boundary.loc[boundary['日落分鐘'].idxmax()]
fixed_end_minute=math.floor(earliest['日出分鐘'])
fixed_start_minute=math.ceil(latest['日落分鐘'])
assert (fixed_start_minute,fixed_end_minute)==(1129,302)
display(Markdown('### 固定時段界線：全年365日 × 資料座標範圍四角'))
display(pd.DataFrame([earliest,latest],index=['最早日出所在列','最晚日落所在列']))
print('固定夜間：18:49（含）至翌日05:02（不含）；臺灣時間UTC+8。')

### 固定時段界線：全年365日 × 資料座標範圍四角

,日期,緯度,經度,日出,日落,日出分鐘,日落分鐘
最早日出所在列,2025-06-09,25.1833,121.6520,2025-06-09T05:02:50.800739+08:00,2025-06-09T18:42:34.298359+08:00,302.8467,"1,122.5716"
最晚日落所在列,2025-07-02,25.1833,121.4612,2025-07-02T05:08:09.382489+08:00,2025-07-02T18:48:12.258035+08:00,308.1564,"1,128.2043"


固定夜間：18:49（含）至翌日05:02（不含）；臺灣時間UTC+8。


In [5]:
out = raw.copy()
out['record_id'] = np.arange(1, len(raw) + 1)
dt = pd.to_datetime(dict(year=raw['發生年度'] + 1911, month=raw['發生月'],
                        day=raw['發生日'], hour=raw['發生時-Hours'], minute=raw['發生分']))
out['crash_datetime'] = dt.dt.strftime('%Y-%m-%d %H:%M:%S')
out['district'] = raw['區序'].str.replace(r'^\d+', '', regex=True)
for source, target, mapping in [('光線','lighting_label',LIGHT),
                                 ('天候','weather_label',WEATHER),
                                 ('道路型態','road_form_label',ROAD)]:
    assert set(raw[source].dropna()).issubset(mapping), f'出現未收錄的 {source} 代碼'
    out[target] = raw[source].map(mapping)
out['fatal24'] = raw['死亡人數'].ge(1).astype('Int64').mask(raw['死亡人數'].isna())
clock_minute = dt.dt.hour*60 + dt.dt.minute
out['night_fixed'] = (clock_minute.ge(fixed_start_minute) | clock_minute.lt(fixed_end_minute)).astype('Int64')
rises, sets, night = [], [], []
for timestamp, lat, lon in zip(dt, raw['座標-Y'], raw['座標-X']):
    if pd.isna(lat) or pd.isna(lon):
        rises.append(None); sets.append(None); night.append(pd.NA)
        continue
    observer = Observer(latitude=float(lat), longitude=float(lon), elevation=0)
    rise = sunrise(observer, date=timestamp.date(), tzinfo=tz)
    setting = sunset(observer, date=timestamp.date(), tzinfo=tz)
    local = timestamp.to_pydatetime().replace(tzinfo=tz)
    rises.append(rise.isoformat()); sets.append(setting.isoformat())
    night.append(int(local < rise or local > setting))
out['sunrise_local'] = rises
out['sunset_local'] = sets
out['is_night'] = pd.array(night, dtype='Int64')
assert len(out) == len(raw) and out['record_id'].is_unique
pd.testing.assert_frame_equal(out[raw.columns], raw)
assert out['is_night'].isna().equals(raw[['座標-X','座標-Y']].isna().any(axis=1))
assert out['fatal24'].isna().equals(raw['死亡人數'].isna())
display(out[['record_id','crash_datetime','district','lighting_label','is_night','fatal24']].head(10))

# 實際入選事故逐筆驗證，不能只靠參考點界線就宣稱保證。
selected_known=out['night_fixed'].eq(1) & out['is_night'].notna()
assert out.loc[selected_known,'is_night'].eq(1).all()
print({'固定夜間筆數':int(out['night_fixed'].sum()),
       '固定夜間但當日不是夜間':int(out.loc[selected_known,'is_night'].ne(1).sum()),
       '固定夜間且缺座標':int((out['night_fixed'].eq(1)&out['is_night'].isna()).sum())})


,record_id,crash_datetime,district,lighting_label,is_night,fatal24
0,1,2025-01-01 00:18:00,中山區,有照明且開啟,1,0
1,2,2025-01-01 00:31:00,文山區,有照明且開啟,1,0
2,3,2025-01-01 00:35:00,中山區,有照明且開啟,1,0
3,4,2025-01-01 00:41:00,中山區,有照明且開啟,1,0
4,5,2025-01-01 00:57:00,南港區,有照明且開啟,1,0
5,6,2025-01-01 01:34:00,信義區,有照明且開啟,1,0
6,7,2025-01-01 02:15:00,信義區,有照明且開啟,1,0
7,8,2025-01-01 06:26:00,文山區,無照明,1,0
8,9,2025-01-01 06:29:00,文山區,有照明未開啟或故障,1,0
9,10,2025-01-01 07:11:00,松山區,有照明且開啟,0,0


{'固定夜間筆數': 5027, '固定夜間但當日不是夜間': 0, '固定夜間且缺座標': 0}


### 5. 寫出資料並重新讀檔驗證

寫出後重新讀取實際磁碟上的 CSV，比較列數、欄位及數值是否一致。
這一步可發現存檔造成的欄位遺漏或數值變動；輸出 SHA-256 則讓下游 Notebook
可以綁定這個已驗證的資料版本。日後若刻意更新輸入，應先核對新版本並更新相關文件。

In [6]:
output_path = project_file('data/processed/taipei_2025_descriptive.csv', 'processed', must_exist=False)
output_path.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(output_path, index=False, encoding='utf-8-sig', lineterminator='\n')
saved = pd.read_csv(output_path, encoding='utf-8-sig')
pd.testing.assert_frame_equal(saved, out, check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12)
assert saved.shape == (22762, 58)
assert saved['record_id'].is_unique
assert saved['is_night'].isna().sum() == 2
verify(raw_path, '61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c')
verify(pdf_path, 'f40ded1ded1c816a38f421bc972cdc1320bbd280e647792207ec037d05fd3670')
print({'path':'data/processed/taipei_2025_descriptive.csv', 'rows':len(saved), 'columns':len(saved.columns),
       'night_counts':saved['is_night'].value_counts(dropna=False).to_dict(),
       'sha256':sha256_file(output_path), 'status':'重新讀檔與原始欄位保留檢查通過'})

C:\Users\user\AppData\Local\Temp\ipykernel_18408\4255670325.py:5: FutureWarning: Mismatched null-like values nan and <NA> found. In a future version, pandas equality-testing functions (e.g. assert_frame_equal) will consider these not-matching and raise.
  pd.testing.assert_frame_equal(saved, out, check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12)


{'path': 'data/processed/taipei_2025_descriptive.csv', 'rows': 22762, 'columns': 58, 'night_counts': {0.0: 15965, 1.0: 6795, nan: 2}, 'sha256': '09c8bae171a8307414aa787ce36f9d5d1f2f1c455dd619e250ac3fae41f15a60', 'status': '重新讀檔與原始欄位保留檢查通過'}


C:\Users\user\AppData\Local\Temp\ipykernel_18408\4255670325.py:5: FutureWarning: Mismatched null-like values nan and None found. In a future version, pandas equality-testing functions (e.g. assert_frame_equal) will consider these not-matching and raise.
  pd.testing.assert_frame_equal(saved, out, check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12)


## 處理結果與下一步

處理後仍為 22,762 列，欄數由 47 增為 58；沒有刪除或補填原始事故紀錄。
日出日落定義下，夜間 6,795 筆、非夜間 15,965 筆，另有 2 筆因座標缺失無法判定。
這 2 筆仍保留在年度概況中，但不納入主要夜間比較。

接著可閱讀 [衍生資料說明](../processed/taipei_2025_descriptive_documentation.ipynb)，
確認各欄的分布與缺值，再閱讀 [敘述性統計](../../finding/敘述性統計.ipynb)。
本次依確認的新固定夜間定義重新執行處理並更新衍生 CSV；原始資料保持原版本。